# 01 — Source Inspection

This notebook documents the initial inspection of the FIPEX historical source.

## Scope

- Load the historical Bronze bootstrap
- Inspect schema, dtypes and row counts
- Validate temporal coverage
- Inspect nulls and basic domains
- Inspect FIPE code structure
- Inspect price consistency
- Establish baseline monthly volumes

This notebook is exploratory. Production validation logic lives in `src/fipe_pipeline/validate.py`.


In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

HISTORICAL_PATH = (
    PROJECT_ROOT / "data" / "bronze" / "historical" / "fipe_history_2026_08.parquet"
)
HISTORICAL_PATH

In [ ]:
df_history = pd.read_parquet(HISTORICAL_PATH)
df_history.shape

## Expected baseline

Historical bootstrap through `2026-08`:

- Rows: `9,478,205`
- Columns: `12`
- Coverage: `2001-01` through `2026-08`
- Distinct reference months: `308`


In [ ]:
df_history.columns.tolist()

In [ ]:
df_history.dtypes

In [ ]:
df_history.head()

## Null inspection


In [ ]:
null_summary = df_history.isna().sum().to_frame("null_count")
null_summary["null_pct"] = null_summary["null_count"] / len(df_history) * 100
null_summary

In [ ]:
pd.crosstab(df_history["zero_km"], df_history["ano_modelo"].isna(), margins=True)

Observed rule:

```text
zero_km = True  -> ano_modelo is null
ano_modelo null -> zero_km = True
```

This relationship is bidirectional in the inspected historical dataset.


## Temporal coverage


In [ ]:
periods = (
    df_history[["ano_referencia", "mes_referencia"]]
    .drop_duplicates()
    .sort_values(["ano_referencia", "mes_referencia"])
    .assign(
        data_referencia=lambda x: pd.to_datetime(
            {"year": x["ano_referencia"], "month": x["mes_referencia"], "day": 1}
        )
    )
)
periods.head(), periods.tail(), len(periods)

In [ ]:
expected_periods = pd.date_range(
    periods["data_referencia"].min(), periods["data_referencia"].max(), freq="MS"
)
missing_periods = expected_periods.difference(periods["data_referencia"])
missing_periods

## Monthly volume baseline


In [ ]:
monthly_volume = (
    df_history.groupby(["ano_referencia", "mes_referencia"])
    .size()
    .rename("rows")
    .reset_index()
)
monthly_volume["data_referencia"] = pd.to_datetime(
    {
        "year": monthly_volume["ano_referencia"],
        "month": monthly_volume["mes_referencia"],
        "day": 1,
    }
)
monthly_volume["mom_pct_change"] = monthly_volume["rows"].pct_change().mul(100)
monthly_volume.tail(12)

In [ ]:
monthly_volume["rows"].describe()

In [ ]:
monthly_volume["mom_pct_change"].describe()

## Domain inspection


In [ ]:
df_history["tipo_veiculo"].value_counts()

In [ ]:
fuel_mapping = (
    df_history[["sigla_combustivel", "nome_combustivel"]]
    .drop_duplicates()
    .sort_values("sigla_combustivel")
)
fuel_mapping

## FIPE code validation


In [ ]:
fipe_code_checks = pd.Series(
    {
        "nulls": df_history["codigo_fipe"].isna().sum(),
        "invalid_length": df_history["codigo_fipe"].str.len().ne(8).sum(),
        "invalid_regex": (
            ~df_history["codigo_fipe"].str.fullmatch(r"\d{6}-\d", na=False)
        ).sum(),
        "leading_or_trailing_spaces": df_history["codigo_fipe"]
        .ne(df_history["codigo_fipe"].str.strip())
        .sum(),
        "distinct_codes": df_history["codigo_fipe"].nunique(),
    }
)
fipe_code_checks

## Model-year temporal rule


In [ ]:
mask = df_history["ano_modelo"].notna()
model_year_gap = (
    df_history.loc[mask, "ano_modelo"] - df_history.loc[mask, "ano_referencia"]
)
model_year_gap.describe()

Structural validation rule:

```text
ano_modelo <= ano_referencia + 1
```

The historical minimum gap is descriptive only and must not be used as a future validation threshold.


## Price validation


In [ ]:
formatted_numeric = (
    df_history["valor_formatado"]
    .str.replace("R$", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .astype(float)
)
price_mismatch = formatted_numeric.ne(df_history["valor_centavos"] / 100)
price_mismatch.sum()

In [ ]:
invalid_price_rows = df_history[df_history["valor_centavos"] <= 0]
invalid_price_rows

## Inspection conclusion

The historical source is structurally consistent overall.

Known issues discovered during inspection:

- Exact duplicates exist
- Non-exact logical-grain collisions exist
- 22 rows contain non-positive prices
- Historical descriptive attributes may change over time
- `ano_modelo` nullability is structurally tied to `zero_km`

Detailed grain and historical cardinality analysis continues in `02_historical_grain_analysis.ipynb`.
